In [1]:
import pathlib
import sys

import duckdb
import pandas as pd

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def load_tables(*names: str) -> dict[str, pd.DataFrame]:
    with duckdb.connect(DB, read_only=True) as con:
        return {name: con.execute(f'SELECT * FROM "{name}"').df() for name in names}


print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [2]:
data = load_tables('income', 'balance', 'cashflow', 'companies', 'industries', 'sec_filings')
for name, df in data.items():
    print(f'{name:12s}: {len(df):,} rows')

income      : 68,495 rows
balance     : 68,486 rows
cashflow    : 68,486 rows
companies   : 6,556 rows
industries  : 74 rows


In [3]:
from irp.quality import run

findings = run(data)
print(f'Total findings: {len(findings):,}')
findings.head()

KeyboardInterrupt: 

In [ ]:
# Summary: findings per rule + severity
(
    findings.groupby(['severity', 'rule'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values(['severity', 'count'], ascending=[True, False])
)

In [ ]:
# Drill-down: accounting identity violations
findings[findings['rule'] == 'accounting_identity'].sort_values(
    'value', ascending=False
)

In [ ]:
# Drill-down: impossible values
findings[findings['rule'] == 'impossible_value']

In [ ]:
# Drill-down: sector outliers
findings[findings['rule'] == 'sector_outlier'].sort_values(
    'value', key=abs, ascending=False
)

In [ ]:
# Drill-down: sudden jumps
findings[findings['rule'] == 'sudden_jump'].sort_values(
    'value', key=abs, ascending=False
)

In [ ]:
# Export all findings for manual review
out = pathlib.Path('../data/flagged_anomalies.csv')
findings.to_csv(out, index=False)
print(f'Exported {len(findings):,} findings → {out.resolve()}')